<a href="https://colab.research.google.com/github/asomers205/DS2002FA26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last_problem = None
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.ConnectionError, requests.Timeout) as e:
            # network trouble is transient -- worth another try
            last_problem = f'{type(e).__name__}: {e}'
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                # 400/404 etc. mean *our* request is wrong -- retrying will not fix it
                raise ValueError(f'{r.status_code} from {r.url}: {r.text[:200]}')
            last_problem = f'HTTP {r.status_code}'
        if attempt < tries:
            delay = 2 ** (attempt - 1)          # 1s, 2s, 4s, ...
            print(f'attempt {attempt} failed ({last_problem}); retrying in {delay}s')
            time.sleep(delay)
    raise RuntimeError(f'get_weather gave up after {tries} attempts; last problem: {last_problem}')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
from datetime import datetime

WANTED = ['2026-09-05', '2026-10-03']
START, END = min(WANTED), max(WANTED)
used_fallback = False

try:
    raw = get_weather(START, END)          # one request covering both dates
    pulled_at = datetime.now()
    wx = raw[raw['time'].isin(WANTED)].reset_index(drop=True)
    print(f'pulled {len(raw)} days, kept {len(wx)}:')
    print(wx)
except RuntimeError as e:
    # only a genuine outage (retries exhausted) lands here; a 400/404 is a bug and should crash
    print('API unavailable, will use the fallback ->', e)
    used_fallback = True
    pulled_at = None

pulled 29 days, kept 2:
         time  temperature_2m_max  precipitation_sum
0  2026-09-05                29.4                2.5
1  2026-10-03                22.0               14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()

wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'
wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = (
    wx.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm',
    })
    .assign(date=lambda d: pd.to_datetime(d['date']))
    [['date', 'temp_max', 'precip_mm', 'source']]
)
print(wx.dtypes)
wx

date         datetime64[ns]
temp_max            float64
precip_mm           float64
source               object
dtype: object


,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
wet = wx.loc[wx['precip_mm'].idxmax()]
dry = wx.loc[wx['precip_mm'].idxmin()]
diff = wet['precip_mm'] - dry['precip_mm']

fmt = lambda d: f'{d:%A, %B} {d.day}, {d.year}'
label = ' (NOTE: these are fallback sample values, not real observations)' if used_fallback else ''

if diff == 0:
    print(f'Both home weekends got the same rainfall: {wet["precip_mm"]:.1f} mm.{label}')
else:
    print(f'{fmt(wet["date"])} was the wetter home game day: {wet["precip_mm"]:.1f} mm of rain '
          f'({wet["precip_mm"] / 25.4:.2f} in) in Charlottesville, versus {dry["precip_mm"]:.1f} mm '
          f'on {fmt(dry["date"])} -- {diff:.1f} mm ({diff / 25.4:.2f} in) more.{label}')

Saturday, October 3, 2026 was the wetter home game day: 14.7 mm of rain (0.58 in) in Charlottesville, versus 2.5 mm on Saturday, September 5, 2026 -- 12.2 mm (0.48 in) more.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
print('Source:      Open-Meteo historical weather archive API')
print('Endpoint:   ', BASE)
print('Coordinates: 38.03, -78.51 (Charlottesville, VA), timezone America/New_York')
print(f'Date range:  {START} to {END} (one request); kept {", ".join(WANTED)}')
print('Variables:   temperature_2m_max (deg C), precipitation_sum (mm)')
print('Pulled:     ', pulled_at.strftime('%Y-%m-%d %H:%M') if pulled_at else 'n/a -- request failed')
print('Data is:    ', 'FALLBACK SAMPLE (not real)' if used_fallback else 'real, from open-meteo')

Source:      Open-Meteo historical weather archive API
Endpoint:    https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51 (Charlottesville, VA), timezone America/New_York
Date range:  2026-09-05 to 2026-10-03 (one request); kept 2026-09-05, 2026-10-03
Variables:   temperature_2m_max (deg C), precipitation_sum (mm)
Pulled:      2026-10-11 13:48
Data is:     real, from open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

**Nothing would break, and that is the problem.**

1. `get_weather` hits a 503 (or my laptop is offline) and quietly hands back `pd.DataFrame()` -- no columns, no rows, no error.
2. The next cells don't crash either. If I `left`-merge weather onto 200 days of sales on `date`, pandas keeps all 200 sales rows and fills every weather column with `NaN`. (An `inner` merge fails differently but just as quietly: it returns 0 rows.)
3. I chart rain against revenue. Matplotlib/seaborn drop `NaN` points without complaint, so I get either a blank scatter or -- if I `fillna(0)` first, which is a very common "cleanup" step -- 200 points stacked on `precip = 0`. That chart says "it never rained, so you can't see any effect of rain on sales." It looks like a finding about the weather, but it's really a finding about the network being down.
4. **Why nobody catches it:** every cell ran, the sales numbers are real, the row count is still 200, and there is no traceback anywhere. The failure moved from an error in the console to a claim in the findings. Anyone reviewing the slide sees a plausible flat relationship, not a missing input, and the "result" won't even reproduce the next time someone runs it with a working network.

That is why `get_weather` raises when every retry fails, why the fallback has to be chosen explicitly and is stamped with a `source` column, and why Part 5 checks the row count and non-null precipitation *before* the join -- an empty frame would trip `len(wx) == 2` immediately.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch: same two dates for a second city, one long-format frame
CITIES = {
    'Charlottesville, VA': (38.03, -78.51),
    'Orlando, FL':         (28.54, -81.38),
}

def clean_weather(raw, city, source):
    """API column names -> ours, keep the wanted dates, tag city + source."""
    return (
        raw[raw['time'].isin(WANTED)]
        .rename(columns={'time': 'date', 'temperature_2m_max': 'temp_max',
                         'precipitation_sum': 'precip_mm'})
        .assign(date=lambda d: pd.to_datetime(d['date']), city=city, source=source)
        [['city', 'date', 'temp_max', 'precip_mm', 'source']]
    )

frames = []
for city, (lat, lon) in CITIES.items():
    try:
        frames.append(clean_weather(get_weather(START, END, lat=lat, lon=lon), city, 'open-meteo'))
    except RuntimeError as e:
        print(f'{city}: API unavailable ({e}); no fallback sample exists for this city, skipping')

both = pd.concat(frames, ignore_index=True)

assert list(both.columns) == ['city', 'date', 'temp_max', 'precip_mm', 'source']
assert len(both) == len(frames) * len(WANTED), 'expected one row per city per date'
assert not both.duplicated(['city', 'date']).any()
assert both['precip_mm'].notna().all() and (both['precip_mm'] >= 0).all()

print(both.to_string(index=False))
print()
print(both.pivot(index='date', columns='city', values='precip_mm').rename_axis(columns=None))

               city       date  temp_max  precip_mm     source
Charlottesville, VA 2026-09-05      29.4        2.5 open-meteo
Charlottesville, VA 2026-10-03      22.0       14.7 open-meteo
        Orlando, FL 2026-09-05      32.1        3.9 open-meteo
        Orlando, FL 2026-10-03      30.4       14.5 open-meteo

            Charlottesville, VA  Orlando, FL
date                                        
2026-09-05                  2.5          3.9
2026-10-03                 14.7         14.5
